In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/latent_fusion')
    if not REPO_ROOT.exists():
        attempt = subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO_ROOT)], capture_output=True)
        if attempt.returncode:
            REPO_ROOT = Path('/content/drive/MyDrive/latent_fusion')
    elif (REPO_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    PANEL_PATH = Path('/content/drive/MyDrive/latent_fusion_data/fusion_vol_panel.npz')
    OUT_DIR = Path('/content/drive/MyDrive/latent_fusion_data/outputs_vol')
else:
    REPO_ROOT = Path(os.getcwd())
    while not ((REPO_ROOT / 'src').exists() and (REPO_ROOT / 'pyproject.toml').exists()) and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    PANEL_PATH = REPO_ROOT / 'data' / 'fusion_vol_panel.npz'
    OUT_DIR = REPO_ROOT / 'outputs' / 'fusion_vol'

if not (REPO_ROOT / 'src' / 'fusion').exists():
    raise FileNotFoundError(f'Módulo não encontrado em {REPO_ROOT}; publique o commit ou copie o repo para o Drive')
if not PANEL_PATH.is_file():
    raise FileNotFoundError(f'Envie o novo painel de volatilidade para {PANEL_PATH}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import torch
from src.fusion.dataset import load_panel_file
from src.fusion.run_tests import run_ablation, plot_volatility, write_report

{'dispositivo': 'cuda' if torch.cuda.is_available() else 'cpu', 'painel': str(PANEL_PATH)}

# Fusão texto × série temporal: volatilidade futura

Experimento de ablação da proposta (`docs/plan.md` §2.1 e §3.1): apenas MOMENT + volatilidade passada, apenas notícias, e atenção cruzada com gate. Alvo: volatilidade realizada anualizada nos **próximos 20 pregões**; persistência dos 20 pregões anteriores como referência. Treino/validação/teste separados por data, com purga de rótulos nas fronteiras. Texto atrasado um pregão; ausência de notícia mantida como máscara.

Antes de executar: ative a GPU T4; gere o novo painel com `uv run python scripts/export_fusion_panel.py`, envie `data/fusion_vol_panel.npz` ao Drive em `MyDrive/latent_fusion_data/`, e publique o código atualizado no GitHub (ou copie o repositório para o Drive). O painel anterior `fusion_panel.npz` contém retorno diário e **não pode ser reutilizado**.

RMSE/MAE/R² medem erro na escala de volatilidade anualizada (0,20 = 20%); não há backtest direcional para uma previsão de volatilidade. As janelas futuras de 20 pregões se sobrepõem: as linhas não são observações independentes, portanto pequenas diferenças não demonstram significância estatística. Entropia e permutação de texto são diagnósticos, não prova causal.

In [ ]:
panel = load_panel_file(PANEL_PATH)
{'tickers': len(panel['tickers']), 'horizonte': panel['horizon'], 'treino': len(panel['splits']['train']), 'validação': len(panel['splits']['val']), 'teste': len(panel['splits']['test'])}

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Ative a GPU T4 antes de iniciar o treino')
result = run_ablation(panel, device='cuda', out_dir=OUT_DIR)
result['history']

## Ablação contra persistência e modalidades isoladas

In [ ]:
result['table']

## Desempenho por regime conhecido e presença de notícias

In [ ]:
result['groups']

## Atenção cruzada e perturbação do texto

In [ ]:
result['attn_summary']

In [ ]:
plot_volatility(result['dailies'], OUT_DIR / 'volatility.png')

In [ ]:
write_report(panel, result, OUT_DIR)
{'relatório': str(OUT_DIR / 'results.md'), 'tabela': str(OUT_DIR / 'results.csv'), 'grupos': str(OUT_DIR / 'groups.csv'), 'figura': str(OUT_DIR / 'volatility.png')}